In [ ]:
import pyspark.sql.functions as F
from lakehouse_common import create_spark, parse_timestamp, valid_coordinates, haversine, require_unique_orders

spark = create_spark("Delivery - Silver")
base = "s3a://lakehouse/warehouse/"
df = spark.read.format("delta").load(base + "delivery_bronze")
require_unique_orders(df)

# Silver keeps every source order. Quality flags are metadata, not deletion rules.
df_clean = (df.withColumn("accept_ts", parse_timestamp("accept_time"))
              .withColumn("delivery_ts", parse_timestamp("delivery_time")))
for name in ["lng", "lat", "accept_gps_lng", "accept_gps_lat", "delivery_gps_lng", "delivery_gps_lat"]:
    df_clean = df_clean.withColumn(name, F.col(name).cast("double"))
for name in ["region_id", "courier_id", "aoi_id", "aoi_type"]:
    df_clean = df_clean.withColumn(name, F.col(name).cast("long"))
df_clean = (df_clean
    .withColumn("delivery_minutes", (F.col("delivery_ts").cast("long") - F.col("accept_ts").cast("long")) / 60.0)
    .withColumn("day_gap", F.datediff(F.to_date("delivery_ts"), F.to_date("accept_ts")))
    .withColumn("distance_km", haversine("lat", "lng", "accept_gps_lat", "accept_gps_lng"))
    .withColumn("delivery_stop_distance_km", haversine("lat", "lng", "delivery_gps_lat", "delivery_gps_lng")))
df_clean = (df_clean
    .withColumn("q_invalid_timestamp", F.col("accept_ts").isNull() | F.col("delivery_ts").isNull())
    .withColumn("q_zero_duration", F.col("delivery_minutes") == 0)
    .withColumn("q_negative_duration", F.col("delivery_minutes") < 0)
    .withColumn("q_cross_day", F.col("day_gap") > 0)
    .withColumn("q_duration_gt24h", F.col("delivery_minutes") > 1440)
    .withColumn("q_accept_gps_invalid", ~valid_coordinates("accept_gps_lat", "accept_gps_lng"))
    .withColumn("q_delivery_gps_invalid", ~valid_coordinates("delivery_gps_lat", "delivery_gps_lng"))
    .withColumn("q_accept_distance_gt50km", F.col("distance_km") > 50)
    .withColumn("q_delivery_distance_gt50km", F.col("delivery_stop_distance_km") > 50)
    .withColumn("eligible_duration", F.col("accept_ts").isNotNull() & F.col("delivery_ts").isNotNull() & (F.col("delivery_minutes") > 0))
    .withColumn("eligible_ml_distance", F.col("eligible_duration") & F.col("distance_km").isNotNull() & ~F.col("q_accept_gps_invalid")))
rejected = (df_clean.filter(~F.col("eligible_duration"))
            .withColumn("reject_reason", F.when(F.col("q_invalid_timestamp"), "invalid_timestamp")
             .when(F.col("q_negative_duration") | F.col("q_zero_duration"), "non_positive_duration")
             .otherwise("duration_not_eligible")))
df_final = df_clean.filter(F.col("eligible_duration"))
before, after = df.count(), df_final.count()
assert after + rejected.count() == before
df_final.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(base + "delivery_silver")
rejected.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(base + "delivery_rejected")
print(f"Bronze={before:,}; Silver positive-duration={after:,}; rejected={before-after:,}")
df_final.groupBy("q_cross_day", "q_duration_gt24h", "q_accept_gps_invalid").count().orderBy("q_cross_day").show(20, False)
